# Model Training
Training CNN, ResNet18 and ResNet34 with augmentation.

In [ ]:
import os
import copy
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from torchvision import transforms

if os.path.basename(os.getcwd()) == 'experiments':
    os.chdir('..')

from src.dataset.data_pipeline import DataPipeline
from src.dataset.trainer import Trainer
from src.models.factory import Factory
from src.dataset.augmentation import Augmentation
from src.models.tta_voting import tta_voting

N_FOLDS = 5
# change if you have GPU memory issues
BATCH_SIZE = 256 

# Windows requires 0 
NUM_WORKERS = 0

EPOCHS = 10
csv_path = f'folds/train_folds_{N_FOLDS}.csv'

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using:", device)

Using: cuda


# Transforms and training loop

In [2]:
transforms_aug = {
    'train': transforms.Compose([
        Augmentation(prob=0.4),
        transforms.ToTensor(),
        transforms.RandomApply([transforms.ElasticTransform(alpha=34.0, sigma=4.0)], p=0.4)
    ]),
    'validation': transforms.Compose([transforms.ToTensor()])
}

def train_and_eval(df, model_name, fold):
    train_df = df[df['fold'] != fold].reset_index(drop=True)
    val_df = df[df['fold'] == fold].reset_index(drop=True)

    pipeline = DataPipeline(train_df, val_df, transforms=transforms_aug)
    train_loader, val_loader = pipeline.get_loaders(batch_size=BATCH_SIZE, num_workers=NUM_WORKERS)

    model = Factory.get_model(model_name, num_classes=10).to(device)

    criterion = torch.nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

    trainer = Trainer(model, criterion, optimizer, device)
    best_acc, history = trainer.fit(train_loader, val_loader, EPOCHS)

    best_state = copy.deepcopy(model.state_dict())
    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

    return best_acc, history, n_params, best_state

# Run training folds and save models
Testing both augmented and baseline datasets.

In [3]:
df = pd.read_csv(csv_path)

# Initialize globals safely so progress isn't wiped if a single cell crashes
if 'results' not in globals():
    results = {}
    histories = {}
    params_count = {}
    best_models_memory = []

m = "CNN"
print(f"\nTraining {m} on GPU...")
fold_scores = []

for fold in range(N_FOLDS):
    acc, hist, n_p, best_state = train_and_eval(df, m, fold)
    fold_scores.append(acc)
    
    # Save fold 0 for soft voting and history plotting
    if fold == 0:
        histories[m] = hist
        params_count[m] = n_p
        best_models_memory.append({'name': m, 'state': best_state})
            
avg_score = sum(fold_scores) / len(fold_scores)
results[m] = avg_score
print(f"Average accuracy for {m}: {avg_score:.4f}")


Training CNN on GPU...
Epoch 01/10 | train_loss=0.8683, train_acc=0.7293 | val_loss=0.1909, val_acc=0.9529
Epoch 02/10 | train_loss=0.2759, train_acc=0.9220 | val_loss=0.0488, val_acc=0.9882
Epoch 03/10 | train_loss=0.1680, train_acc=0.9515 | val_loss=0.0344, val_acc=0.9912
Epoch 04/10 | train_loss=0.1273, train_acc=0.9637 | val_loss=0.0217, val_acc=0.9935
Epoch 05/10 | train_loss=0.0962, train_acc=0.9735 | val_loss=0.0232, val_acc=0.9941
Epoch 06/10 | train_loss=0.1006, train_acc=0.9699 | val_loss=0.0166, val_acc=0.9956
Epoch 07/10 | train_loss=0.0903, train_acc=0.9714 | val_loss=0.0142, val_acc=0.9965
Epoch 08/10 | train_loss=0.0801, train_acc=0.9772 | val_loss=0.0146, val_acc=0.9962


KeyboardInterrupt: 

Training CNN on GPU...
Epoch 01/10 | train_loss=1.0759, train_acc=0.6643 | val_loss=2.1018, val_acc=0.3956
Epoch 02/10 | train_loss=0.3722, train_acc=0.8982 | val_loss=0.2741, val_acc=0.9126
Epoch 03/10 | train_loss=0.2121, train_acc=0.9406 | val_loss=0.0805, val_acc=0.9771
Epoch 04/10 | train_loss=0.1599, train_acc=0.9555 | val_loss=0.0515, val_acc=0.9841
Epoch 05/10 | train_loss=0.1271, train_acc=0.9618 | val_loss=0.0258, val_acc=0.9935
Epoch 06/10 | train_loss=0.1052, train_acc=0.9685 | val_loss=0.0196, val_acc=0.9947
Epoch 07/10 | train_loss=0.0897, train_acc=0.9756 | val_loss=0.0225, val_acc=0.9938
Epoch 08/10 | train_loss=0.0897, train_acc=0.9728 | val_loss=0.0236, val_acc=0.9944
Epoch 09/10 | train_loss=0.0643, train_acc=0.9818 | val_loss=0.0149, val_acc=0.9965
Epoch 10/10 | train_loss=0.0636, train_acc=0.9810 | val_loss=0.0136, val_acc=0.9968
Epoch 01/10 | train_loss=1.0552, train_acc=0.6710 | val_loss=2.3245, val_acc=0.2376
Epoch 02/10 | train_loss=0.3588, train_acc=0.9009 | val_loss=0.5603, val_acc=0.8218
Epoch 03/10 | train_loss=0.2169, train_acc=0.9401 | val_loss=0.1304, val_acc=0.9662
Epoch 04/10 | train_loss=0.1517, train_acc=0.9572 | val_loss=0.0277, val_acc=0.9944
Epoch 05/10 | train_loss=0.1213, train_acc=0.9668 | val_loss=0.0308, val_acc=0.9926
Epoch 06/10 | train_loss=0.1002, train_acc=0.9721 | val_loss=0.0138, val_acc=0.9965
Epoch 07/10 | train_loss=0.0841, train_acc=0.9754 | val_loss=0.0149, val_acc=0.9968
Epoch 08/10 | train_loss=0.0790, train_acc=0.9768 | val_loss=0.0251, val_acc=0.9909
Epoch 09/10 | train_loss=0.0699, train_acc=0.9792 | val_loss=0.0098, val_acc=0.9971
Epoch 10/10 | train_loss=0.0594, train_acc=0.9833 | val_loss=0.0098, val_acc=0.9971
Epoch 01/10 | train_loss=1.0862, train_acc=0.6596 | val_loss=1.1847, val_acc=0.7021
Epoch 02/10 | train_loss=0.3701, train_acc=0.9017 | val_loss=0.1142, val_acc=0.9668
Epoch 03/10 | train_loss=0.2154, train_acc=0.9396 | val_loss=0.0691, val_acc=0.9800
...
Epoch 10/10 | train_loss=0.0599, train_acc=0.9821 | val_loss=0.0130, val_acc=0.9965
Average accuracy for CNN: 0.9970

In [4]:
m = "ResNet18"
print(f"\nTraining {m} on GPU...")
fold_scores = []

for fold in range(N_FOLDS):
    acc, hist, n_p, best_state = train_and_eval(df, m, fold)
    fold_scores.append(acc)
    
    # Save fold 0 for soft voting and history plotting
    if fold == 0:
        histories[m] = hist
        params_count[m] = n_p
        best_models_memory.append({'name': m, 'state': best_state})
            
avg_score = sum(fold_scores) / len(fold_scores)
results[m] = avg_score
print(f"Average accuracy for {m}: {avg_score:.4f}")


Training ResNet18 on GPU...
Epoch 01/10 | train_loss=0.4117, train_acc=0.8685 | val_loss=0.1535, val_acc=0.9503
Epoch 02/10 | train_loss=0.0832, train_acc=0.9746 | val_loss=0.0204, val_acc=0.9953
Epoch 03/10 | train_loss=0.0444, train_acc=0.9865 | val_loss=0.0175, val_acc=0.9941
Epoch 04/10 | train_loss=0.0596, train_acc=0.9816 | val_loss=0.0171, val_acc=0.9959
Epoch 05/10 | train_loss=0.0311, train_acc=0.9904 | val_loss=0.0116, val_acc=0.9968
Epoch 06/10 | train_loss=0.0282, train_acc=0.9911 | val_loss=0.0140, val_acc=0.9962
Epoch 07/10 | train_loss=0.0260, train_acc=0.9920 | val_loss=0.0114, val_acc=0.9971
Epoch 08/10 | train_loss=0.0242, train_acc=0.9925 | val_loss=0.0136, val_acc=0.9974
Epoch 09/10 | train_loss=0.0211, train_acc=0.9930 | val_loss=0.0309, val_acc=0.9894
Epoch 10/10 | train_loss=0.0188, train_acc=0.9942 | val_loss=0.0450, val_acc=0.9894
Epoch 01/10 | train_loss=0.3822, train_acc=0.8739 | val_loss=0.0743, val_acc=0.9765
Epoch 02/10 | train_loss=0.0815, train_acc=0.97

In [5]:
m = "ResNet34"
print(f"\nTraining {m} on GPU...")
fold_scores = []

for fold in range(N_FOLDS):
    acc, hist, n_p, best_state = train_and_eval(df, m, fold)
    fold_scores.append(acc)
    
    # Save fold 0 for soft voting and history plotting
    if fold == 0:
        histories[m] = hist
        params_count[m] = n_p
        best_models_memory.append({'name': m, 'state': best_state})
            
avg_score = sum(fold_scores) / len(fold_scores)
results[m] = avg_score
print(f"Average accuracy for {m}: {avg_score:.4f}")


Training ResNet34 on GPU...
Epoch 01/10 | train_loss=0.4743, train_acc=0.8421 | val_loss=0.0740, val_acc=0.9721
Epoch 02/10 | train_loss=0.0934, train_acc=0.9703 | val_loss=0.0324, val_acc=0.9906
Epoch 03/10 | train_loss=0.0516, train_acc=0.9832 | val_loss=0.0569, val_acc=0.9800


KeyboardInterrupt: 

# Parameters and Accuracy Comparison

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(10, 4))
m_names = list(results.keys())

# Parameter comparison
p_vals = [params_count[m] for m in m_names]
axs[0].bar(m_names, p_vals)
axs[0].set_title("Network Parameters")

# Accuracy comparison
aug_acc = [results[m] for m in m_names]

axs[1].bar(m_names, aug_acc, width=0.5, color='orange')
axs[1].set_title("Validation Accuracy (Augmented Data)")
axs[1].set_ylim([0.95, 1.0])

plt.show()

# Training History Graphs

In [ ]:
for m in m_names:
    h = histories[m]
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3))
    
    ax1.plot(h['train_acc'], label='train', color='blue')
    ax1.plot(h['val_acc'], label='val', color='orange')
    ax1.set_title(f"{m} Accuracy Overview")
    ax1.legend()
    
    ax2.plot(h['train_loss'], label='train', color='red')
    ax2.plot(h['val_loss'], label='val', color='green')
    ax2.set_title(f"{m} Loss Decrease")
    ax2.legend()
    
    plt.show()

# Soft Voting Ensemble
Combine predictions from the best augmented models using soft decision voting.

In [ ]:
fold = 0
val_df = df[df['fold'] == fold].reset_index(drop=True)

transforms_base = {
    'train': transforms.Compose([transforms.ToTensor()]),
    'validation': transforms.Compose([transforms.ToTensor()])
}

pipeline = DataPipeline(df[df['fold'] != fold], val_df, transforms=transforms_base)
_, val_loader = pipeline.get_loaders(batch_size=BATCH_SIZE, num_workers=NUM_WORKERS)

loaded_networks = []
for saved in best_models_memory:
    m_name = saved['name']
    net = Factory.get_model(m_name, num_classes=10).to(device)
    net.load_state_dict(saved['state'])
    loaded_networks.append(net)

correct = 0
total = 0

for images, labels in val_loader:
    images = images.to(device)
    labels = labels.to(device)
    
    preds = tta_voting(loaded_networks, images, device)
    correct += (preds == labels).sum().item()
    total += labels.size(0)

acc = correct / total
print(f"Final testing accuracy with TTA and Soft Voting: {acc:.4f}")